In [ ]:
import sys
import os
import argparse
import numpy as np
import pandas as pd

In [ ]:
NARROWPEAK_SCHEMA = ["chr", "start", "end", "1", "2", "3", "4", "5", "6", "summit"]
X_peaks_coords = pd.read_csv("~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C3Cardiomyocytes/data/region_for_predict/vista/heart/hs_mm_ht/filter_common_name/heart_tissue_VISTA_HsToTS_05.filter.10col.bed", sep='\t', header=None, names = NARROWPEAK_SCHEMA)
X_peaks_coords.head()

In [ ]:
X_peaks_coords['2']=X_peaks_coords['chr'] + ":" + (X_peaks_coords['start']+1).astype(str) + "-" + X_peaks_coords['end'].astype(str)
X_peaks_coords['3']=X_peaks_coords['end']-X_peaks_coords['start'] #length
X_peaks_coords.head()

In [ ]:
X_peaks_coords['3'].describe()

In [ ]:
X_peaks_coords_indexed = X_peaks_coords.set_index("1")
X_peaks_coords_indexed.head()

In [ ]:
C3_1_motif_matrix=pd.read_csv("~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C3Cardiomyocytes/output/predict/C3.1/finemo/heart_tissue_VISTA_HsToTS_05.filter/counts/TS_C3.1_VISTA_HsToTS_05.filter_motif_presence_matrix.csv")
C3_2_motif_matrix=pd.read_csv("~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C3Cardiomyocytes/output/predict/C3.2/finemo/heart_tissue_VISTA_HsToTS_05.filter/counts/TS_C3.2_VISTA_HsToTS_05.filter_motif_presence_matrix.csv")

In [ ]:
C3_1_motif_matrix

In [ ]:
type(C3_1_motif_matrix)

In [ ]:
C3_1_motif_matrix = C3_1_motif_matrix.set_index("original_peak_name")
C3_1_motif_matrix.head()

In [ ]:
C3_1_row_sums = C3_1_motif_matrix.sum(axis=1)
C3_1_row_sums_per_1000bp=C3_1_row_sums / (X_peaks_coords_indexed.loc[C3_1_motif_matrix.index, '3'] / 1000)
C3_1_row_sums_per_1000bp.describe()

In [ ]:
C3_1_row_sums_per_1000bp.max()
C3_1_row_sums_per_1000bp.idxmax()

In [ ]:
C3_2_motif_matrix

In [ ]:
C3_2_motif_matrix = C3_2_motif_matrix.set_index("original_peak_name")
C3_2_motif_matrix.head()

In [ ]:
C3_2_row_sums = C3_2_motif_matrix.sum(axis=1)
C3_2_row_sums_per_1000bp=C3_2_row_sums / (X_peaks_coords_indexed.loc[C3_2_motif_matrix.index, '3'] / 1000)
C3_2_row_sums_per_1000bp.describe()

In [ ]:
C3_1_row_sums_per_1000bp.max()
C3_1_row_sums_per_1000bp.idxmax()

In [ ]:
# ========== 导入绘图库 ==========
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# ========== 合并数据用于绘图 ==========
# 将两个 Series 转换为 DataFrame，并添加分组标签
#reset_index(drop=True)：重置索引，将原来的 peak 名称索引丢弃，变成默认的整数索引（0,1,2,...）。这样做的原因是后续合并两个样本时，我们不需要保留 peak 名称，只需要密度的数值和对应的样本标签
df_c31 = C3_1_row_sums_per_1000bp.reset_index(drop=True).to_frame(name='motif_density_per_1000bp')
df_c31['group'] = 'C3.1'

df_c32 = C3_2_row_sums_per_1000bp.reset_index(drop=True).to_frame(name='motif_density_per_1000bp')
df_c32['group'] = 'C3.2'

# 合并两个 DataFrame。按行拼接。ignore_index=True 让合并后的 DataFrame 重新生成整数索引，避免索引冲突
combined_df = pd.concat([df_c31, df_c32], ignore_index=True)

# 检查数据
print(combined_df.groupby('group')['motif_density_per_1000bp'].describe())

In [ ]:
combined_df

In [ ]:
# ========== 绘制小提琴图 ==========
plt.figure(figsize=(8, 6))

#当 inner='quartile' 时，会在小提琴内部绘制三条水平线，分别对应数据的 第一四分位数 (Q1)、中位数 (Q2) 和 第三四分位数 (Q3)。这样能直观地看到每组数据的中心趋势和四分位范围，而无需额外的箱线图。
#palette='Set2'指定调色板，用于给不同组的小提琴着色。'Set2' 是 seaborn 内置的一种定性调色板，颜色柔和且区分度高，适合分组较少的分类数据。
#外部形状表示数据的核密度估计，即数据在 y 轴不同值上的分布概率。形状越宽，表明该值附近的数据点越多。内部的四分位线（由 inner='quartile' 提供）标记了四分位数位置，帮助判断分布的离散程度。
sns.violinplot(x='group', y='motif_density_per_1000bp', data=combined_df,
               order=['C3.1', 'C3.2'], inner='quartile', palette='Set2')

# 添加标题和标签
plt.xlabel('Sample Group')
plt.ylabel('Motif density per 1000 bp')
plt.title('Comparison of Motif Density Between C3.1 and C3.2')

# 显示图形
plt.tight_layout()
plt.show()


In [ ]:
# 设置图形大小
plt.figure(figsize=(8, 6))

# 绘制箱线图，showfliers=False 隐藏离群值。箱线图通常会将超出上下须范围（一般为 Q1 - 1.5×IQR 和 Q3 + 1.5×IQR）的数据点单独绘制为散点。设置 False 后这些离群点不会显示，图形只包含箱体、须和中位数等核心统计元素
#箱体：从 Q1 到 Q3，表示中间 50% 数据的范围。箱内横线：中位数（Q2）。须（whiskers）：从箱体延伸至非离群值范围内的最大值和最小值（默认 1.5×IQR 内）。离群值：落在须范围外的点（此处因 showfliers=False 被隐藏）。
sns.boxplot(x='group', y='motif_density_per_1000bp', data=combined_df,
            order=['C3.1', 'C3.2'], showfliers=False, palette='Set2')

# 添加标题和标签
plt.xlabel('Sample Group')
plt.ylabel('Motif density per 1000 bp')
plt.title('Comparison of Motif Density (Boxplot, no outliers)')

# 显示图形
plt.tight_layout()
plt.savefig('~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C3Cardiomyocytes/C3_1_C3_2_vista_motif_per_1000bp_boxplot.pdf')
plt.show()